Imports

In [1]:
import sys
import os
package_path = os.path.abspath("../..")  
sys.path.insert(0, package_path)
#The path will be managed by conda or whatever on release, but this is fine for now.
import scMPRAforge as scm
import pandas as pd
import numpy as np

2025-05-28 17:10:33.701441: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-28 17:10:33.706470: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-05-28 17:10:33.706484: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [1]:
#load the autoreload extension
%load_ext autoreload
#reload code on every execution
#(this may break objects)
#you can remove this & do dev in a notebook, then paste into the module when you are done.
%autoreload 2

In [2]:
#dask imports
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
import socket

In [3]:
scm.helloworld()

hello world!


Make the dask cluster & client in accordance with resource avail and model size

In [4]:
cluster=SLURMCluster(
    cores=2,#cores per slurm job
    memory="16G",#memory per slurm job
    processes=1,#dask workers per slurm job
    job_extra_directives=["-p ycga", 
        f"--job-name=simclust_worker",
        f"--time=2:00:00",
        f"--output=slave_%j.out"]
)
client = Client(cluster,
    timeout=f"{5*60}s",   # Client <-> scheduler timeout 
    heartbeat_interval="20s"  # Worker heartbeat interval
)

In [5]:
cluster.scale(jobs=1)

In [6]:
dat=scm.load_scMPRA_data("/gpfs/gibbs/pi/reilly/tabula_data/simulated/fake_cres.tsv")

In [7]:
#let's make an ortho object
test=scm.ortho()
test.criss_cross(client=client,dat=dat,retain_design_matricies=True)
test.extract_params(client)

In [8]:
description=scm.describe_parameters(client,parameters=test.by_cre_parameters.result(),dat=dat,split="cre_id")

In [9]:
index=description.index

In [10]:
description=scm.auto_partition(description,50)

In [13]:
scm.simulate_from_description(description)


,C(cell_type)[blood],C(cell_type)[brain],C(rep_id)[1],C(rep_id)[2],C(rep_id)[3],cre_id,cells,nb,zi,theta,r,sigmasquare,p,zinb_sample
npartitions=2,,,,,,,,,,,,,,
,int64,int64,int64,int64,int64,string,int64,float64,float64,float32,float32,float64,float64,int64
,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [112]:
type(working_exploded.compute())

pandas.core.frame.DataFrame

In [18]:
cluster.close()